# Pretrain IndPatchTST


In [ ]:
import importlib.util
from pathlib import Path
import sys

# Fallback: if package not installed in this kernel, add repo root to sys.path
if importlib.util.find_spec('src') is None:
    ROOT = Path('..').resolve()
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))



**Prerequisite:** install the package in editable mode from the repo root:

```bash
pip install -e .
```


In [ ]:
from pathlib import Path
import torch
import torch.nn as nn
from src.data.dataloader import build_etth1_dataloaders
from src.models.indpatchtst import build_model_from_config
from src.training.optuna_search import bayesian_search
from src.training.train_indpatchtst_reg import train_and_valid_loop

data_path = Path('../data/ETTh1.csv')
if not data_path.exists():
    raise FileNotFoundError('Place ETTh1.csv in ../data/')

WINDOW, HORIZON = 36, 24
train_dl, val_dl, n_features = build_etth1_dataloaders(
    str(data_path), window=WINDOW, horizon=HORIZON, batch_size=128
)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Recherche bayesienne (reduis n_trials pour un test rapide)
best_params, best_loss = bayesian_search(
    train_dl, val_dl, WINDOW, HORIZON, device, n_trials=10, max_epochs=5
)

# Re-entrainement long avec la meilleure config
model = build_model_from_config(best_params, n_features, WINDOW, HORIZON).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=best_params['lr'], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=20)
criterion = nn.MSELoss()
train_and_valid_loop(
    model, train_dl, val_dl, opt, criterion, n_epochs=10, device=device, scheduler=scheduler
)
